# 05 - Modelagem da Camada Gold

## Objetivo

Construir tabelas analíticas com granularidade semanal para integrar as séries de Brent, PPI e preços de combustíveis da ANP.

A granularidade semanal foi escolhida por ser compatível com a periodicidade do PPI e permitir a comparação entre as diferentes fontes. Além das séries semanais, esta etapa constrói a base integrada utilizada nas análises e calcula correlações entre as variações do Brent, do PPI e dos preços ao consumidor, considerando defasagens de até quatro semanas.

In [0]:
from pyspark.sql.functions import (
    col,
    avg,
    min,
    max,
    count,
    date_trunc
)

# Carregando as tabelas Silver persistidas

brent_silver = spark.table("workspace.silver.brent")
ppi_silver = spark.table("workspace.silver.ppi")
anp_silver = spark.table("workspace.silver.precos_anp")

print("Brent:", brent_silver.count())
print("ANP:", anp_silver.count())

brent_silver.printSchema()

Brent: 9973
ANP: 1374810
root
 |-- data: date (nullable = true)
 |-- preco_brent_usd: double (nullable = true)



In [0]:
from pyspark.sql.functions import (
    col,
    avg,
    min,
    max,
    count,
    date_trunc
)

brent_gold_semanal = (
    brent_silver
        .withColumn(
            "semana",
            date_trunc("week", col("data")).cast("date")
        )
        .groupBy("semana")
        .agg(
            avg("preco_brent_usd").alias("brent_medio_usd"),
            min("preco_brent_usd").alias("brent_min_usd"),
            max("preco_brent_usd").alias("brent_max_usd"),
            count("*").alias("dias_observados")
        )
        .orderBy("semana")
)

In [0]:
print(f"Semanas Brent: {brent_gold_semanal.count():,}")

display(
    brent_gold_semanal
        .orderBy(col("semana").desc())
        .limit(15)
)

Semanas Brent: 2,052


semana,brent_medio_usd,brent_min_usd,brent_max_usd,dias_observados
2026-09-07,106.7,104.47,109.51,3
2026-08-31,99.0925,96.02,102.24,4
2026-08-24,89.72999999999999,87.77,92.71,5
2026-08-17,94.20200000000001,92.37,96.92,5
2026-08-10,92.51399999999998,92.02,93.26,5
2026-08-03,87.85799999999999,86.47,89.65,5
2026-07-27,91.62799999999999,85.51,96.95,5
2026-07-20,96.118,86.99,105.32,5
2026-07-13,82.926,81.23,85.01,5
2026-07-06,73.328,69.56,76.5,5


In [0]:
print(f"Registros PPI: {ppi_silver.count():,}")

ppi_silver.printSchema()

display(
    ppi_silver
        .orderBy(col("data_inicio").desc(), col("produto"), col("localidade"))
        .limit(20)
)

Registros PPI: 12,142
root
 |-- data_inicio: date (nullable = true)
 |-- data_fim: date (nullable = true)
 |-- localidade: string (nullable = true)
 |-- preco: double (nullable = true)
 |-- variacao_semanal: double (nullable = true)
 |-- produto: string (nullable = true)
 |-- variacao_semanal_pct: double (nullable = true)



data_inicio,data_fim,localidade,preco,variacao_semanal,produto,variacao_semanal_pct
2026-08-31,2026-09-04,Aratu,6.22937,0.05832378536993765,DIESEL,5.832378536993765
2026-08-31,2026-09-04,Araucaria,6.30117,0.057715711279098025,DIESEL,5.7715711279098025
2026-08-31,2026-09-04,Betim,6.436979999999999,0.056140515526806656,DIESEL,5.614051552680666
2026-08-31,2026-09-04,Canoas,6.285638,0.057825990224896096,DIESEL,5.7825990224896096
2026-08-31,2026-09-04,Cubatao,6.270916000000001,0.05789893834176718,DIESEL,5.789893834176718
2026-08-31,2026-09-04,Duque_de_Caxias,6.415362,0.056364007600780974,DIESEL,5.636400760078097
2026-08-31,2026-09-04,Guamare,6.341976,0.05676492073733219,DIESEL,5.676492073733219
2026-08-31,2026-09-04,Itaqui,6.23937,0.058224865639361134,DIESEL,5.822486563936113
2026-08-31,2026-09-04,Manaus,6.288600000000001,0.05790145699909677,DIESEL,5.790145699909677
2026-08-31,2026-09-04,Maua,6.288482,0.057708299413461006,DIESEL,5.770829941346101


In [0]:
from pyspark.sql.functions import (
    avg,
    min,
    max,
    count,
    countDistinct
)

ppi_gold_semanal = (
    ppi_silver
        .groupBy(
            "data_inicio",
            "data_fim",
            "produto"
        )
        .agg(
            avg("preco").alias("ppi_medio"),
            min("preco").alias("ppi_min"),
            max("preco").alias("ppi_max"),
            countDistinct("localidade").alias("localidades_observadas")
        )
        .orderBy(
            "data_inicio",
            "produto"
        )
)

In [0]:
print(f"Registros PPI Gold semanal: {ppi_gold_semanal.count():,}")

display(
    ppi_gold_semanal
        .orderBy(
            col("data_inicio").desc(),
            col("produto")
        )
        .limit(20)
)

Registros PPI Gold semanal: 818


data_inicio,data_fim,produto,ppi_medio,ppi_min,ppi_max,localidades_observadas
2026-08-31,2026-09-04,DIESEL,6.290048625,6.188046,6.436979999999999,16
2026-08-31,2026-09-04,GASOLINA,3.6398353750000005,3.56902,3.7864180000000003,16
2026-08-24,2026-08-28,DIESEL,5.947009249999999,5.84642,6.094814,16
2026-08-24,2026-08-28,GASOLINA,3.5324707500000003,3.4615859999999996,3.6801239999999997,16
2026-08-17,2026-08-21,DIESEL,6.109502750000001,6.007024,6.2577359999999995,16
2026-08-17,2026-08-21,GASOLINA,3.5684145000000003,3.497044,3.716348,16
2026-08-10,2026-08-14,DIESEL,5.7894380000000005,5.69397,5.937270000000001,16
2026-08-10,2026-08-14,GASOLINA,3.4824391249999995,3.4119699999999997,3.630764,16
2026-08-03,2026-08-07,DIESEL,5.38436225,5.285626000000001,5.530756,16
2026-08-03,2026-08-07,GASOLINA,3.3443706250000003,3.27404,3.490774,16


In [0]:
display(
    ppi_silver
        .select("localidade")
        .distinct()
        .orderBy("localidade")
)

localidade
Aratu
Araucaria
Betim
Canoas
Cubatao
Duque_de_Caxias
Guamare
Itaqui
Manaus
Maua


In [0]:
print(
    "Localidades distintas:",
    ppi_silver.select("localidade").distinct().count()
)

Localidades distintas: 16


In [0]:
from pyspark.sql.functions import (
    col,
    avg,
    min,
    max,
    count,
    countDistinct,
    date_trunc
)

anp_gold_semanal = (
    anp_silver
        .filter(
            col("produto").isin("GASOLINA", "DIESEL S10")
        )
        .withColumn(
            "semana",
            date_trunc("week", col("data_coleta")).cast("date")
        )
        .groupBy(
            "semana",
            "produto"
        )
        .agg(
            avg("valor_venda").alias("preco_medio_anp"),
            min("valor_venda").alias("preco_min_anp"),
            max("valor_venda").alias("preco_max_anp"),
            count("*").alias("observacoes"),
            countDistinct("uf").alias("ufs_observadas")
        )
        .orderBy(
            "semana",
            "produto"
        )
)

In [0]:
print(
    f"Registros ANP Gold semanal: "
    f"{anp_gold_semanal.count():,}"
)

display(
    anp_gold_semanal
        .orderBy(
            col("semana").desc(),
            col("produto")
        )
        .limit(20)
)

Registros ANP Gold semanal: 176


semana,produto,preco_medio_anp,preco_min_anp,preco_max_anp,observacoes,ufs_observadas
2026-08-31,DIESEL S10,6.937380636604777,5.63,8.99,1508,26
2026-08-31,GASOLINA,6.556554021894337,5.44,8.99,2101,26
2026-08-24,DIESEL S10,6.914838396897216,5.74,9.27,3094,27
2026-08-24,GASOLINA,6.548775510204077,5.39,8.99,4361,27
2026-08-17,DIESEL S10,6.923696816892528,5.77,9.19,3173,27
2026-08-17,GASOLINA,6.560445289773992,5.49,8.99,4469,27
2026-08-10,DIESEL S10,6.954237018158645,5.77,9.27,3139,27
2026-08-10,GASOLINA,6.57642552235452,5.49,8.99,4451,27
2026-08-03,DIESEL S10,6.976342637151106,5.84,9.19,3117,27
2026-08-03,GASOLINA,6.59284098620221,5.49,8.99,4421,27


In [0]:
from pyspark.sql.functions import when

ppi_integracao = (
    ppi_gold_semanal
        .withColumnRenamed("data_inicio", "semana")
        .withColumn(
            "combustivel",
            when(col("produto") == "GASOLINA", "GASOLINA")
            .when(col("produto") == "DIESEL", "DIESEL")
        )
        .select(
            "semana",
            "combustivel",
            "ppi_medio",
            "ppi_min",
            "ppi_max",
            "localidades_observadas"
        )
)

anp_integracao = (
    anp_gold_semanal
        .withColumn(
            "combustivel",
            when(col("produto") == "GASOLINA", "GASOLINA")
            .when(col("produto") == "DIESEL S10", "DIESEL")
        )
        .select(
            "semana",
            "combustivel",
            "preco_medio_anp",
            "preco_min_anp",
            "preco_max_anp",
            "observacoes",
            "ufs_observadas"
        )
)

In [0]:
print("PPI integração:", ppi_integracao.count())
print("ANP integração:", anp_integracao.count())

display(
    ppi_integracao
        .orderBy(col("semana").desc(), "combustivel")
        .limit(10)
)

display(
    anp_integracao
        .orderBy(col("semana").desc(), "combustivel")
        .limit(10)
)

PPI integração: 818
ANP integração: 176


semana,combustivel,ppi_medio,ppi_min,ppi_max,localidades_observadas
2026-08-31,DIESEL,6.290048625,6.188046,6.436979999999999,16
2026-08-31,GASOLINA,3.6398353750000005,3.56902,3.7864180000000003,16
2026-08-24,DIESEL,5.947009249999999,5.84642,6.094814,16
2026-08-24,GASOLINA,3.5324707500000003,3.4615859999999996,3.6801239999999997,16
2026-08-17,DIESEL,6.109502750000001,6.007024,6.2577359999999995,16
2026-08-17,GASOLINA,3.5684145000000003,3.497044,3.716348,16
2026-08-10,DIESEL,5.7894380000000005,5.69397,5.937270000000001,16
2026-08-10,GASOLINA,3.4824391249999995,3.4119699999999997,3.630764,16
2026-08-03,DIESEL,5.38436225,5.285626000000001,5.530756,16
2026-08-03,GASOLINA,3.3443706250000003,3.27404,3.490774,16


semana,combustivel,preco_medio_anp,preco_min_anp,preco_max_anp,observacoes,ufs_observadas
2026-08-31,DIESEL,6.937380636604777,5.63,8.99,1508,26
2026-08-31,GASOLINA,6.556554021894337,5.44,8.99,2101,26
2026-08-24,DIESEL,6.914838396897216,5.74,9.27,3094,27
2026-08-24,GASOLINA,6.548775510204077,5.39,8.99,4361,27
2026-08-17,DIESEL,6.923696816892528,5.77,9.19,3173,27
2026-08-17,GASOLINA,6.560445289773992,5.49,8.99,4469,27
2026-08-10,DIESEL,6.954237018158645,5.77,9.27,3139,27
2026-08-10,GASOLINA,6.57642552235452,5.49,8.99,4451,27
2026-08-03,DIESEL,6.976342637151106,5.84,9.19,3117,27
2026-08-03,GASOLINA,6.59284098620221,5.49,8.99,4421,27


In [0]:
from pyspark.sql.functions import min, max, countDistinct

print("BRENT")
display(
    brent_gold_semanal.agg(
        min("semana").alias("inicio"),
        max("semana").alias("fim"),
        countDistinct("semana").alias("semanas")
    )
)

print("PPI")
display(
    ppi_integracao.agg(
        min("semana").alias("inicio"),
        max("semana").alias("fim"),
        countDistinct("semana").alias("semanas")
    )
)

print("ANP")
display(
    anp_integracao.agg(
        min("semana").alias("inicio"),
        max("semana").alias("fim"),
        countDistinct("semana").alias("semanas")
    )
)

BRENT


inicio,fim,semanas
1987-05-18,2026-09-07,2052


PPI


inicio,fim,semanas
2018-11-05,2026-08-31,411


ANP


inicio,fim,semanas
2024-12-30,2026-08-31,88


In [0]:
anp_ppi = (
    anp_integracao
        .join(
            ppi_integracao,
            on=["semana", "combustivel"],
            how="left"
        )
        .orderBy("semana", "combustivel")
)

print("Registros ANP:", anp_integracao.count())
print("Registros após join ANP + PPI:", anp_ppi.count())

anp_ppi.printSchema()

display(
    anp_ppi
        .orderBy(col("semana").desc(), "combustivel")
        .limit(20)
)

Registros ANP: 176
Registros após join ANP + PPI: 176
root
 |-- semana: date (nullable = true)
 |-- combustivel: string (nullable = true)
 |-- preco_medio_anp: double (nullable = true)
 |-- preco_min_anp: double (nullable = true)
 |-- preco_max_anp: double (nullable = true)
 |-- observacoes: long (nullable = false)
 |-- ufs_observadas: long (nullable = false)
 |-- ppi_medio: double (nullable = true)
 |-- ppi_min: double (nullable = true)
 |-- ppi_max: double (nullable = true)
 |-- localidades_observadas: long (nullable = true)



semana,combustivel,preco_medio_anp,preco_min_anp,preco_max_anp,observacoes,ufs_observadas,ppi_medio,ppi_min,ppi_max,localidades_observadas
2026-08-31,DIESEL,6.937380636604777,5.63,8.99,1508,26,6.290048625,6.188046,6.436979999999999,16
2026-08-31,GASOLINA,6.556554021894337,5.44,8.99,2101,26,3.6398353750000005,3.56902,3.7864180000000003,16
2026-08-24,DIESEL,6.914838396897216,5.74,9.27,3094,27,5.947009249999999,5.84642,6.094814,16
2026-08-24,GASOLINA,6.548775510204077,5.39,8.99,4361,27,3.5324707500000003,3.4615859999999996,3.6801239999999997,16
2026-08-17,DIESEL,6.923696816892528,5.77,9.19,3173,27,6.109502750000001,6.007024,6.2577359999999995,16
2026-08-17,GASOLINA,6.560445289773992,5.49,8.99,4469,27,3.5684145000000003,3.497044,3.716348,16
2026-08-10,DIESEL,6.954237018158645,5.77,9.27,3139,27,5.7894380000000005,5.69397,5.937270000000001,16
2026-08-10,GASOLINA,6.57642552235452,5.49,8.99,4451,27,3.4824391249999995,3.4119699999999997,3.630764,16
2026-08-03,DIESEL,6.976342637151106,5.84,9.19,3117,27,5.38436225,5.285626000000001,5.530756,16
2026-08-03,GASOLINA,6.59284098620221,5.49,8.99,4421,27,3.3443706250000003,3.27404,3.490774,16


In [0]:
from pyspark.sql.functions import col, sum, when

display(
    anp_ppi.select(
        sum(
            when(col("ppi_medio").isNull(), 1).otherwise(0)
        ).alias("registros_sem_ppi")
    )
)

registros_sem_ppi
0


In [0]:
base_integrada = (
    anp_ppi
        .join(
            brent_gold_semanal,
            on="semana",
            how="left"
        )
)

In [0]:
print("Registros antes do Brent:", anp_ppi.count())
print("Registros após o Brent:", base_integrada.count())

display(
    base_integrada.select(
        sum(
            when(col("brent_medio_usd").isNull(), 1).otherwise(0)
        ).alias("registros_sem_brent")
    )
)

display(
    base_integrada
        .orderBy(col("semana").desc(), "combustivel")
        .limit(20)
)

Registros antes do Brent: 176
Registros após o Brent: 176


registros_sem_brent
0


semana,combustivel,preco_medio_anp,preco_min_anp,preco_max_anp,observacoes,ufs_observadas,ppi_medio,ppi_min,ppi_max,localidades_observadas,brent_medio_usd,brent_min_usd,brent_max_usd,dias_observados
2026-08-31,DIESEL,6.937380636604777,5.63,8.99,1508,26,6.290048625,6.188046,6.436979999999999,16,99.0925,96.02,102.24,4
2026-08-31,GASOLINA,6.556554021894337,5.44,8.99,2101,26,3.6398353750000005,3.56902,3.7864180000000003,16,99.0925,96.02,102.24,4
2026-08-24,DIESEL,6.914838396897216,5.74,9.27,3094,27,5.947009249999999,5.84642,6.094814,16,89.72999999999999,87.77,92.71,5
2026-08-24,GASOLINA,6.548775510204077,5.39,8.99,4361,27,3.5324707500000003,3.4615859999999996,3.6801239999999997,16,89.72999999999999,87.77,92.71,5
2026-08-17,DIESEL,6.923696816892528,5.77,9.19,3173,27,6.109502750000001,6.007024,6.2577359999999995,16,94.20200000000001,92.37,96.92,5
2026-08-17,GASOLINA,6.560445289773992,5.49,8.99,4469,27,3.5684145000000003,3.497044,3.716348,16,94.20200000000001,92.37,96.92,5
2026-08-10,DIESEL,6.954237018158645,5.77,9.27,3139,27,5.7894380000000005,5.69397,5.937270000000001,16,92.51399999999998,92.02,93.26,5
2026-08-10,GASOLINA,6.57642552235452,5.49,8.99,4451,27,3.4824391249999995,3.4119699999999997,3.630764,16,92.51399999999998,92.02,93.26,5
2026-08-03,DIESEL,6.976342637151106,5.84,9.19,3117,27,5.38436225,5.285626000000001,5.530756,16,87.85799999999999,86.47,89.65,5
2026-08-03,GASOLINA,6.59284098620221,5.49,8.99,4421,27,3.3443706250000003,3.27404,3.490774,16,87.85799999999999,86.47,89.65,5


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, lag

janela_combustivel = (
    Window
        .partitionBy("combustivel")
        .orderBy("semana")
)

base_variacoes = (
    base_integrada

    # valores da semana anterior
    .withColumn(
        "brent_anterior",
        lag("brent_medio_usd").over(janela_combustivel)
    )
    .withColumn(
        "ppi_anterior",
        lag("ppi_medio").over(janela_combustivel)
    )
    .withColumn(
        "anp_anterior",
        lag("preco_medio_anp").over(janela_combustivel)
    )

    # variação percentual Brent
    .withColumn(
        "var_brent_pct",
        ((col("brent_medio_usd") / col("brent_anterior")) - 1) * 100
    )

    # variação percentual PPI
    .withColumn(
        "var_ppi_pct",
        ((col("ppi_medio") / col("ppi_anterior")) - 1) * 100
    )

    # variação percentual preço ANP
    .withColumn(
        "var_anp_pct",
        ((col("preco_medio_anp") / col("anp_anterior")) - 1) * 100
    )
)

In [0]:
display(
    base_variacoes
        .select(
            "semana",
            "combustivel",
            "brent_medio_usd",
            "var_brent_pct",
            "ppi_medio",
            "var_ppi_pct",
            "preco_medio_anp",
            "var_anp_pct"
        )
        .orderBy(col("semana").desc(), "combustivel")
        .limit(30)
)

semana,combustivel,brent_medio_usd,var_brent_pct,ppi_medio,var_ppi_pct,preco_medio_anp,var_anp_pct
2026-08-31,DIESEL,99.0925,10.434080017831281,6.290048625000001,5.768267049525777,6.937380636604743,0.3259980698556708
2026-08-31,GASOLINA,99.0925,10.434080017831281,3.639835375,3.0393634540356596,6.556554021894312,0.1187781086416706
2026-08-24,DIESEL,89.72999999999999,-4.747245281416557,5.947009250000001,-2.659684538156548,6.91483839689722,-0.12794349939905247
2026-08-24,GASOLINA,89.72999999999999,-4.747245281416557,3.5324707500000003,-1.007275079730774,6.548775510204103,-0.17788090677482993
2026-08-17,DIESEL,94.20200000000001,1.8245887108978343,6.10950275,5.52842521156629,6.923696816892533,-0.4391596257989905
2026-08-17,GASOLINA,94.20200000000001,1.8245887108978343,3.5684145,2.4688263574456526,6.5604452897740195,-0.24299267932416102
2026-08-10,DIESEL,92.51399999999998,5.299460493068353,5.7894380000000005,7.523189027632782,6.954237018158653,-0.31686544285720597
2026-08-10,GASOLINA,92.51399999999998,5.299460493068353,3.482439125,4.128385142720248,6.576425522354547,-0.24898922758843645
2026-08-03,DIESEL,87.85799999999999,-4.114462827956511,5.384362249999999,-7.003751357732268,6.976342637151098,-0.1626497833571272
2026-08-03,GASOLINA,87.85799999999999,-4.114462827956511,3.344370625,-8.798124051143875,6.5928409862022255,-0.19062568078658249


### Análise de defasagens temporais

Para investigar se as variações do Brent e do PPI aparecem nos preços domésticos com atraso, são consideradas defasagens de até quatro semanas.

O lag 0 representa a associação entre variações ocorridas na mesma semana, enquanto os lags de 1 a 4 representam as variações do Brent ou do PPI ocorridas de uma a quatro semanas antes da variação observada na série comparada.

As correlações são utilizadas como medida de associação entre as séries e não, isoladamente, como evidência de causalidade.

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, lag

janela = (
    Window
        .partitionBy("combustivel")
        .orderBy("semana")
)

base_lags = (
    base_variacoes

    # Brent de 1 a 4 semanas atrás
    .withColumn("brent_lag1", lag("var_brent_pct", 1).over(janela))
    .withColumn("brent_lag2", lag("var_brent_pct", 2).over(janela))
    .withColumn("brent_lag3", lag("var_brent_pct", 3).over(janela))
    .withColumn("brent_lag4", lag("var_brent_pct", 4).over(janela))

    # PPI de 1 a 4 semanas atrás
    .withColumn("ppi_lag1", lag("var_ppi_pct", 1).over(janela))
    .withColumn("ppi_lag2", lag("var_ppi_pct", 2).over(janela))
    .withColumn("ppi_lag3", lag("var_ppi_pct", 3).over(janela))
    .withColumn("ppi_lag4", lag("var_ppi_pct", 4).over(janela))
)

In [0]:
display(
    base_lags
        .select(
            "semana",
            "combustivel",

            "var_brent_pct",
            "brent_lag1",
            "brent_lag2",
            "brent_lag3",
            "brent_lag4",

            "var_ppi_pct",
            "ppi_lag1",
            "ppi_lag2",
            "ppi_lag3",
            "ppi_lag4",

            "var_anp_pct"
        )
        .orderBy(col("semana").desc(), "combustivel")
        .limit(20)
)

semana,combustivel,var_brent_pct,brent_lag1,brent_lag2,brent_lag3,brent_lag4,var_ppi_pct,ppi_lag1,ppi_lag2,ppi_lag3,ppi_lag4,var_anp_pct
2026-08-31,DIESEL,10.434080017831281,-4.747245281416557,1.8245887108978343,5.299460493068353,-4.114462827956511,5.768267049525777,-2.659684538156548,5.52842521156629,7.523189027632782,-7.003751357732268,0.3259980698556708
2026-08-31,GASOLINA,10.434080017831281,-4.747245281416557,1.8245887108978343,5.299460493068353,-4.114462827956511,3.0393634540356596,-1.007275079730774,2.4688263574456526,4.128385142720248,-8.798124051143875,0.1187781086416706
2026-08-24,DIESEL,-4.747245281416557,1.8245887108978343,5.299460493068353,-4.114462827956511,-4.671341476102297,-2.659684538156548,5.52842521156629,7.523189027632782,-7.003751357732268,3.2174596330920346,-0.12794349939905247
2026-08-24,GASOLINA,-4.747245281416557,1.8245887108978343,5.299460493068353,-4.114462827956511,-4.671341476102297,-1.007275079730774,2.4688263574456526,4.128385142720248,-8.798124051143875,-2.06185697796869,-0.17788090677482993
2026-08-17,DIESEL,1.8245887108978343,5.299460493068353,-4.114462827956511,-4.671341476102297,15.9081590815908,5.52842521156629,7.523189027632782,-7.003751357732268,3.2174596330920346,6.254782043926732,-0.4391596257989905
2026-08-17,GASOLINA,1.8245887108978343,5.299460493068353,-4.114462827956511,-4.671341476102297,15.9081590815908,2.4688263574456526,4.128385142720248,-8.798124051143875,-2.06185697796869,3.908187503167615,-0.24299267932416102
2026-08-10,DIESEL,5.299460493068353,-4.114462827956511,-4.671341476102297,15.9081590815908,13.089133755182193,7.523189027632782,-7.003751357732268,3.2174596330920346,6.254782043926732,13.084999890887271,-0.31686544285720597
2026-08-10,GASOLINA,5.299460493068353,-4.114462827956511,-4.671341476102297,15.9081590815908,13.089133755182193,4.128385142720248,-8.798124051143875,-2.06185697796869,3.908187503167615,8.652411639868053,-0.24898922758843645
2026-08-03,DIESEL,-4.114462827956511,-4.671341476102297,15.9081590815908,13.089133755182193,5.205164992826372,-7.003751357732268,3.2174596330920346,6.254782043926732,13.084999890887271,5.692485503459754,-0.1626497833571272
2026-08-03,GASOLINA,-4.114462827956511,-4.671341476102297,15.9081590815908,13.089133755182193,5.205164992826372,-8.798124051143875,-2.06185697796869,3.908187503167615,8.652411639868053,3.5096428332606733,-0.19062568078658249


In [0]:
from pyspark.sql.functions import col, corr, lit

resultados = []

for combustivel in ["GASOLINA", "DIESEL"]:

    df = base_lags.filter(col("combustivel") == combustivel)

    for lag_num in range(0, 5):

        coluna_brent = (
            "var_brent_pct"
            if lag_num == 0
            else f"brent_lag{lag_num}"
        )

        correlacao = (
            df
            .select(
                corr(
                    col(coluna_brent),
                    col("var_ppi_pct")
                ).alias("correlacao")
            )
            .first()["correlacao"]
        )

        resultados.append(
            (
                combustivel,
                "Brent → PPI",
                lag_num,
                correlacao
            )
        )

correlacoes_brent_ppi = spark.createDataFrame(
    resultados,
    ["combustivel", "relacao", "lag_semanas", "correlacao"]
)

display(
    correlacoes_brent_ppi
        .orderBy("combustivel", "lag_semanas")
)

combustivel,relacao,lag_semanas,correlacao
DIESEL,Brent → PPI,0,0.7811291323758789
DIESEL,Brent → PPI,1,0.27912019856877635
DIESEL,Brent → PPI,2,-0.0980769807775201
DIESEL,Brent → PPI,3,0.07962631475263526
DIESEL,Brent → PPI,4,-0.09479426251938823
GASOLINA,Brent → PPI,0,0.7468862415611741
GASOLINA,Brent → PPI,1,0.30423714860882817
GASOLINA,Brent → PPI,2,-0.1331199979223173
GASOLINA,Brent → PPI,3,0.03974318323242233
GASOLINA,Brent → PPI,4,0.018930590122846282


In [0]:
resultados = []

for combustivel in ["GASOLINA", "DIESEL"]:

    df = base_lags.filter(col("combustivel") == combustivel)

    for lag_num in range(0, 5):

        coluna_ppi = (
            "var_ppi_pct"
            if lag_num == 0
            else f"ppi_lag{lag_num}"
        )

        correlacao = (
            df
            .select(
                corr(
                    col(coluna_ppi),
                    col("var_anp_pct")
                ).alias("correlacao")
            )
            .first()["correlacao"]
        )

        resultados.append(
            (
                combustivel,
                "PPI → ANP",
                lag_num,
                correlacao
            )
        )

correlacoes_ppi_anp = spark.createDataFrame(
    resultados,
    ["combustivel", "relacao", "lag_semanas", "correlacao"]
)

display(
    correlacoes_ppi_anp
        .orderBy("combustivel", "lag_semanas")
)

combustivel,relacao,lag_semanas,correlacao
DIESEL,PPI → ANP,0,0.34042536569226767
DIESEL,PPI → ANP,1,0.6555478676836615
DIESEL,PPI → ANP,2,0.4112189132646083
DIESEL,PPI → ANP,3,0.22967983645353884
DIESEL,PPI → ANP,4,-0.02001700028107129
GASOLINA,PPI → ANP,0,0.3151432140655167
GASOLINA,PPI → ANP,1,0.4789470988299642
GASOLINA,PPI → ANP,2,0.3533337127899155
GASOLINA,PPI → ANP,3,0.10813589345402597
GASOLINA,PPI → ANP,4,-0.07047963433644287


In [0]:
resultados = []

for combustivel in ["GASOLINA", "DIESEL"]:

    df = base_lags.filter(col("combustivel") == combustivel)

    for lag_num in range(0, 5):

        coluna_brent = (
            "var_brent_pct"
            if lag_num == 0
            else f"brent_lag{lag_num}"
        )

        correlacao = (
            df
            .select(
                corr(
                    col(coluna_brent),
                    col("var_anp_pct")
                ).alias("correlacao")
            )
            .first()["correlacao"]
        )

        resultados.append(
            (
                combustivel,
                "Brent → ANP",
                lag_num,
                correlacao
            )
        )

correlacoes_brent_anp = spark.createDataFrame(
    resultados,
    ["combustivel", "relacao", "lag_semanas", "correlacao"]
)

display(
    correlacoes_brent_anp
        .orderBy("combustivel", "lag_semanas")
)

combustivel,relacao,lag_semanas,correlacao
DIESEL,Brent → ANP,0,0.32147463430314105
DIESEL,Brent → ANP,1,0.47000729303341177
DIESEL,Brent → ANP,2,0.2356179494638037
DIESEL,Brent → ANP,3,0.11496080278812118
DIESEL,Brent → ANP,4,0.02731372632786952
GASOLINA,Brent → ANP,0,0.18668524845187998
GASOLINA,Brent → ANP,1,0.40792614657465526
GASOLINA,Brent → ANP,2,0.30295808029482785
GASOLINA,Brent → ANP,3,0.15135560858191935
GASOLINA,Brent → ANP,4,-0.008470543869503258


In [0]:
correlacoes = (
    correlacoes_brent_ppi
        .unionByName(correlacoes_ppi_anp)
        .unionByName(correlacoes_brent_anp)
)

display(
    correlacoes
        .orderBy(
            "combustivel",
            "relacao",
            "lag_semanas"
        )
)

combustivel,relacao,lag_semanas,correlacao
DIESEL,Brent → ANP,0,0.32147463430314105
DIESEL,Brent → ANP,1,0.47000729303341177
DIESEL,Brent → ANP,2,0.2356179494638037
DIESEL,Brent → ANP,3,0.11496080278812118
DIESEL,Brent → ANP,4,0.02731372632786952
DIESEL,Brent → PPI,0,0.7811291323758789
DIESEL,Brent → PPI,1,0.27912019856877635
DIESEL,Brent → PPI,2,-0.0980769807775201
DIESEL,Brent → PPI,3,0.07962631475263526
DIESEL,Brent → PPI,4,-0.09479426251938823


In [0]:
from pyspark.sql.functions import col, corr

def calcular_correlacoes(df, x_base, x_prefixo, y, relacao):

    resultados = []

    for combustivel in ["GASOLINA", "DIESEL"]:

        dados = df.filter(col("combustivel") == combustivel)

        for lag_num in range(0, 5):

            coluna_x = (
                x_base
                if lag_num == 0
                else f"{x_prefixo}{lag_num}"
            )

            dados_validos = (
                dados
                .select(coluna_x, y)
                .dropna()
            )

            n = dados_validos.count()

            r = (
                dados_validos
                .select(
                    corr(
                        col(coluna_x),
                        col(y)
                    ).alias("r")
                )
                .first()["r"]
            )

            resultados.append(
                (
                    combustivel,
                    relacao,
                    lag_num,
                    r,
                    n
                )
            )

    return spark.createDataFrame(
        resultados,
        [
            "combustivel",
            "relacao",
            "lag_semanas",
            "correlacao",
            "n"
        ]
    )

In [0]:
corr_brent_ppi = calcular_correlacoes(
    base_lags,
    "var_brent_pct",
    "brent_lag",
    "var_ppi_pct",
    "Brent → PPI"
)

corr_ppi_anp = calcular_correlacoes(
    base_lags,
    "var_ppi_pct",
    "ppi_lag",
    "var_anp_pct",
    "PPI → ANP"
)

corr_brent_anp = calcular_correlacoes(
    base_lags,
    "var_brent_pct",
    "brent_lag",
    "var_anp_pct",
    "Brent → ANP"
)

correlacoes_final = (
    corr_brent_ppi
    .unionByName(corr_ppi_anp)
    .unionByName(corr_brent_anp)
)

display(
    correlacoes_final
    .orderBy(
        "combustivel",
        "relacao",
        "lag_semanas"
    )
)

combustivel,relacao,lag_semanas,correlacao,n
DIESEL,Brent → ANP,0,0.32147463430314105,87
DIESEL,Brent → ANP,1,0.47000729303341177,86
DIESEL,Brent → ANP,2,0.2356179494638037,85
DIESEL,Brent → ANP,3,0.11496080278812118,84
DIESEL,Brent → ANP,4,0.02731372632786952,83
DIESEL,Brent → PPI,0,0.7811291323758789,87
DIESEL,Brent → PPI,1,0.27912019856877635,86
DIESEL,Brent → PPI,2,-0.0980769807775201,85
DIESEL,Brent → PPI,3,0.07962631475263526,84
DIESEL,Brent → PPI,4,-0.09479426251938823,83


### Série histórica Brent × PPI

Para analisar a relação entre o Brent e o PPI, é utilizada uma janela histórica mais ampla, aproveitando todo o período comum disponível entre essas duas fontes, sem restringir a análise ao período coberto pelos dados de preços da ANP.

Essa série é utilizada especificamente nas correlações entre as variações semanais do Brent e do PPI. As análises que envolvem preços ao consumidor permanecem restritas à janela comum entre Brent, PPI e ANP.

In [0]:
brent_ppi_historico = (
    ppi_integracao
    .join(
        brent_gold_semanal,
        on="semana",
        how="inner"
    )
)

print(
    "Registros Brent × PPI:",
    brent_ppi_historico.count()
)

display(
    brent_ppi_historico
    .groupBy("combustivel")
    .agg(
        min("semana").alias("inicio"),
        max("semana").alias("fim"),
        count("*").alias("semanas")
    )
)

Registros Brent × PPI: 816


combustivel,inicio,fim,semanas
DIESEL,2018-11-05,2026-08-31,407
GASOLINA,2018-11-05,2026-08-31,409


Versão inicial — não executar da célula 32 até a célular 35: o uso de lag() diretamente sobre o histórico assume continuidade semanal. Foram identificadas duas semanas ausentes para DIESEL (19/11/2018 e 15/04/2019), portanto a abordagem foi substituída pela construção de um calendário semanal completo.

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, lag

janela_hist = (
    Window
    .partitionBy("combustivel")
    .orderBy("semana")
)

historico_variacoes = (
    brent_ppi_historico

    # semana anterior
    .withColumn(
        "brent_anterior",
        lag("brent_medio_usd", 1).over(janela_hist)
    )
    .withColumn(
        "ppi_anterior",
        lag("ppi_medio", 1).over(janela_hist)
    )

    # variação percentual semanal
    .withColumn(
        "var_brent_pct",
        (
            (col("brent_medio_usd") - col("brent_anterior"))
            / col("brent_anterior")
        ) * 100
    )
    .withColumn(
        "var_ppi_pct",
        (
            (col("ppi_medio") - col("ppi_anterior"))
            / col("ppi_anterior")
        ) * 100
    )
)

display(
    historico_variacoes
    .orderBy(col("semana").desc(), "combustivel")
    .select(
        "semana",
        "combustivel",
        "brent_medio_usd",
        "var_brent_pct",
        "ppi_medio",
        "var_ppi_pct"
    )
    .limit(20)
)

semana,combustivel,brent_medio_usd,var_brent_pct,ppi_medio,var_ppi_pct
2026-08-31,DIESEL,99.0925,10.434080017831285,6.290048625000001,5.768267049525776
2026-08-31,GASOLINA,99.0925,10.434080017831285,3.639835375,3.0393634540356707
2026-08-24,DIESEL,89.72999999999999,-4.7472452814165536,5.947009250000001,-2.6596845381565495
2026-08-24,GASOLINA,89.72999999999999,-4.7472452814165536,3.5324707500000003,-1.007275079730774
2026-08-17,DIESEL,94.20200000000001,1.8245887108978436,6.10950275,5.528425211566292
2026-08-17,GASOLINA,94.20200000000001,1.8245887108978436,3.5684145,2.4688263574456553
2026-08-10,DIESEL,92.51399999999998,5.299460493068351,5.7894380000000005,7.523189027632786
2026-08-10,GASOLINA,92.51399999999998,5.299460493068351,3.482439125,4.128385142720243
2026-08-03,DIESEL,87.85799999999999,-4.114462827956515,5.384362249999999,-7.003751357732267
2026-08-03,GASOLINA,87.85799999999999,-4.114462827956515,3.344370625,-8.798124051143878


In [0]:
historico_lags = historico_variacoes

for i in range(1, 5):
    historico_lags = historico_lags.withColumn(
        f"brent_lag{i}",
        lag("var_brent_pct", i).over(janela_hist)
    )

display(
    historico_lags
    .orderBy(col("semana").desc(), "combustivel")
    .select(
        "semana",
        "combustivel",
        "var_brent_pct",
        "brent_lag1",
        "brent_lag2",
        "brent_lag3",
        "brent_lag4",
        "var_ppi_pct"
    )
    .limit(20)
)

semana,combustivel,var_brent_pct,brent_lag1,brent_lag2,brent_lag3,brent_lag4,var_ppi_pct
2026-08-31,DIESEL,10.434080017831285,-4.7472452814165536,1.8245887108978436,5.299460493068351,-4.114462827956515,5.768267049525776
2026-08-31,GASOLINA,10.434080017831285,-4.7472452814165536,1.8245887108978436,5.299460493068351,-4.114462827956515,3.0393634540356707
2026-08-24,DIESEL,-4.7472452814165536,1.8245887108978436,5.299460493068351,-4.114462827956515,-4.671341476102301,-2.6596845381565495
2026-08-24,GASOLINA,-4.7472452814165536,1.8245887108978436,5.299460493068351,-4.114462827956515,-4.671341476102301,-1.007275079730774
2026-08-17,DIESEL,1.8245887108978436,5.299460493068351,-4.114462827956515,-4.671341476102301,15.908159081590806,5.528425211566292
2026-08-17,GASOLINA,1.8245887108978436,5.299460493068351,-4.114462827956515,-4.671341476102301,15.908159081590806,2.4688263574456553
2026-08-10,DIESEL,5.299460493068351,-4.114462827956515,-4.671341476102301,15.908159081590806,13.089133755182193,7.523189027632786
2026-08-10,GASOLINA,5.299460493068351,-4.114462827956515,-4.671341476102301,15.908159081590806,13.089133755182193,4.128385142720243
2026-08-03,DIESEL,-4.114462827956515,-4.671341476102301,15.908159081590806,13.089133755182193,5.2051649928263775,-7.003751357732267
2026-08-03,GASOLINA,-4.114462827956515,-4.671341476102301,15.908159081590806,13.089133755182193,5.2051649928263775,-8.798124051143878


In [0]:
corr_brent_ppi_historica = calcular_correlacoes(
    historico_lags,
    "var_brent_pct",
    "brent_lag",
    "var_ppi_pct",
    "Brent → PPI | 2018–2026"
)

display(
    corr_brent_ppi_historica
    .orderBy("combustivel", "lag_semanas")
)

combustivel,relacao,lag_semanas,correlacao,n
DIESEL,Brent → PPI | 2018–2026,0,0.5920365803350004,406
DIESEL,Brent → PPI | 2018–2026,1,0.2756311431579396,405
DIESEL,Brent → PPI | 2018–2026,2,0.0698160243726906,404
DIESEL,Brent → PPI | 2018–2026,3,0.005327479052802822,403
DIESEL,Brent → PPI | 2018–2026,4,0.0068088938738714855,402
GASOLINA,Brent → PPI | 2018–2026,0,0.6903602071689033,408
GASOLINA,Brent → PPI | 2018–2026,1,0.3248773696609679,407
GASOLINA,Brent → PPI | 2018–2026,2,-0.017487095292935868,406
GASOLINA,Brent → PPI | 2018–2026,3,-0.018928490041430168,405
GASOLINA,Brent → PPI | 2018–2026,4,-0.05639041079288966,404


In [0]:
from pyspark.sql.functions import datediff, lag, col
from pyspark.sql.window import Window

w_gap = (
    Window
    .partitionBy("combustivel")
    .orderBy("semana")
)

validacao_semanas = (
    brent_ppi_historico
    .withColumn(
        "semana_anterior",
        lag("semana").over(w_gap)
    )
    .withColumn(
        "dias_entre_semanas",
        datediff(col("semana"), col("semana_anterior"))
    )
)

display(
    validacao_semanas
    .filter(
        col("dias_entre_semanas").isNotNull() &
        (col("dias_entre_semanas") != 7)
    )
    .select(
        "combustivel",
        "semana_anterior",
        "semana",
        "dias_entre_semanas"
    )
    .orderBy("combustivel", "semana")
)

combustivel,semana_anterior,semana,dias_entre_semanas
DIESEL,2018-11-12,2018-11-26,14
DIESEL,2019-04-08,2019-04-22,14


Voltar a executar a partir da célula 37, abaixo.

In [0]:
from pyspark.sql.functions import (
    col, explode, sequence, lit, expr,
    min as spark_min, max as spark_max
)

# Limites temporais do histórico Brent x PPI
limites = (
    brent_ppi_historico
    .agg(
        spark_min("semana").alias("inicio"),
        spark_max("semana").alias("fim")
    )
    .first()
)

inicio = limites["inicio"]
fim = limites["fim"]

# Calendário semanal completo
calendario = (
    spark.range(1)
    .select(
        explode(
            sequence(
                lit(inicio),
                lit(fim),
                expr("INTERVAL 7 DAYS")
            )
        ).alias("semana")
    )
)

# Um calendário para cada combustível
combustiveis = (
    brent_ppi_historico
    .select("combustivel")
    .distinct()
)

calendario_combustivel = (
    calendario
    .crossJoin(combustiveis)
)

print("Semanas do calendário:", calendario.count())
print(
    "Registros calendário × combustível:",
    calendario_combustivel.count()
)

Semanas do calendário: 409
Registros calendário × combustível: 818


In [0]:
from pyspark.sql.functions import col

brent_ppi_calendario = (
    calendario_combustivel
    .join(
        brent_ppi_historico,
        on=["semana", "combustivel"],
        how="left"
    )
    .orderBy("combustivel", "semana")
)

print("Registros esperados:", calendario_combustivel.count())
print("Registros após o join:", brent_ppi_calendario.count())

display(
    brent_ppi_calendario
    .filter(col("ppi_medio").isNull())
    .select(
        "semana",
        "combustivel",
        "brent_medio_usd",
        "ppi_medio"
    )
    .orderBy("semana")
)

Registros esperados: 818
Registros após o join: 818


semana,combustivel,brent_medio_usd,ppi_medio
2018-11-19,DIESEL,null,null
2019-04-15,DIESEL,null,null


In [0]:
from pyspark.sql.functions import col

# Mantemos do calendário apenas os dados do PPI
ppi_calendario = (
    brent_ppi_calendario
    .select(
        "semana",
        "combustivel",
        "ppi_medio"
    )
)

# Recolocamos o Brent a partir da série Gold original
brent_ppi_calendario_corrigido = (
    ppi_calendario
    .join(
        brent_gold_semanal.select(
            "semana",
            "brent_medio_usd"
        ),
        on="semana",
        how="left"
    )
)

print(
    "Registros:",
    brent_ppi_calendario_corrigido.count()
)

display(
    brent_ppi_calendario_corrigido
    .filter(col("ppi_medio").isNull())
    .orderBy("semana")
)

Registros: 818


semana,combustivel,ppi_medio,brent_medio_usd
2018-11-19,DIESEL,null,61.217999999999996
2019-04-15,DIESEL,null,70.87249999999999


In [0]:
from pyspark.sql.functions import col, lag
from pyspark.sql.window import Window

# Janela cronológica por combustível
w = (
    Window
    .partitionBy("combustivel")
    .orderBy("semana")
)

serie_corrigida = (
    brent_ppi_calendario_corrigido

    # Valores da semana imediatamente anterior
    .withColumn(
        "brent_anterior",
        lag("brent_medio_usd", 1).over(w)
    )
    .withColumn(
        "ppi_anterior",
        lag("ppi_medio", 1).over(w)
    )

    # Variação semanal do Brent
    .withColumn(
        "var_brent_pct",
        (
            (col("brent_medio_usd") - col("brent_anterior"))
            / col("brent_anterior")
        ) * 100
    )

    # Variação semanal do PPI
    .withColumn(
        "var_ppi_pct",
        (
            (col("ppi_medio") - col("ppi_anterior"))
            / col("ppi_anterior")
        ) * 100
    )
)

display(
    serie_corrigida
    .filter(
        (col("semana") >= "2018-11-05") &
        (col("semana") <= "2018-12-03") &
        (col("combustivel") == "DIESEL")
    )
    .select(
        "semana",
        "combustivel",
        "brent_medio_usd",
        "var_brent_pct",
        "ppi_medio",
        "ppi_anterior",
        "var_ppi_pct"
    )
    .orderBy("semana")
)

semana,combustivel,brent_medio_usd,var_brent_pct,ppi_medio,ppi_anterior,var_ppi_pct
2018-11-05,DIESEL,70.344,null,2.269182,null,null
2018-11-12,DIESEL,66.208,-5.879677015808023,2.1791376,2.269182,-3.9681435865435226
2018-11-19,DIESEL,61.217999999999996,-7.536853552440796,null,2.1791376,null
2018-11-26,DIESEL,58.65,-4.194844653533271,2.0116776,null,null
2018-12-03,DIESEL,60.465999999999994,3.0963341858482445,2.0175799999999997,2.0116776,0.29340685604888384


In [0]:
from pyspark.sql.functions import lag
from pyspark.sql.window import Window

w = (
    Window
    .partitionBy("combustivel")
    .orderBy("semana")
)

serie_lags_corrigida = serie_corrigida

for i in range(1, 5):
    serie_lags_corrigida = (
        serie_lags_corrigida
        .withColumn(
            f"brent_lag{i}",
            lag("var_brent_pct", i).over(w)
        )
    )

display(
    serie_lags_corrigida
    .select(
        "semana",
        "combustivel",
        "var_brent_pct",
        "brent_lag1",
        "brent_lag2",
        "brent_lag3",
        "brent_lag4",
        "var_ppi_pct"
    )
    .orderBy(col("semana").desc(), "combustivel")
    .limit(20)
)

semana,combustivel,var_brent_pct,brent_lag1,brent_lag2,brent_lag3,brent_lag4,var_ppi_pct
2026-08-31,DIESEL,10.434080017831285,-4.7472452814165536,1.8245887108978436,5.299460493068351,-4.114462827956515,5.768267049525776
2026-08-31,GASOLINA,10.434080017831285,-4.7472452814165536,1.8245887108978436,5.299460493068351,-4.114462827956515,3.0393634540356707
2026-08-24,DIESEL,-4.7472452814165536,1.8245887108978436,5.299460493068351,-4.114462827956515,-4.671341476102301,-2.6596845381565495
2026-08-24,GASOLINA,-4.7472452814165536,1.8245887108978436,5.299460493068351,-4.114462827956515,-4.671341476102301,-1.007275079730774
2026-08-17,DIESEL,1.8245887108978436,5.299460493068351,-4.114462827956515,-4.671341476102301,15.908159081590806,5.528425211566292
2026-08-17,GASOLINA,1.8245887108978436,5.299460493068351,-4.114462827956515,-4.671341476102301,15.908159081590806,2.4688263574456553
2026-08-10,DIESEL,5.299460493068351,-4.114462827956515,-4.671341476102301,15.908159081590806,13.089133755182193,7.523189027632786
2026-08-10,GASOLINA,5.299460493068351,-4.114462827956515,-4.671341476102301,15.908159081590806,13.089133755182193,4.128385142720243
2026-08-03,DIESEL,-4.114462827956515,-4.671341476102301,15.908159081590806,13.089133755182193,5.2051649928263775,-7.003751357732267
2026-08-03,GASOLINA,-4.114462827956515,-4.671341476102301,15.908159081590806,13.089133755182193,5.2051649928263775,-8.798124051143878


In [0]:
from pyspark.sql.functions import corr, col

resultados_corrigidos = []

for combustivel in ["DIESEL", "GASOLINA"]:

    df_comb = (
        serie_lags_corrigida
        .filter(col("combustivel") == combustivel)
    )

    # Lag 0
    df_valido = df_comb.filter(
        col("var_brent_pct").isNotNull() &
        col("var_ppi_pct").isNotNull()
    )

    r = (
        df_valido
        .select(corr("var_brent_pct", "var_ppi_pct"))
        .first()[0]
    )

    resultados_corrigidos.append(
        (combustivel, "Brent → PPI | 2018–2026", 0, r, df_valido.count())
    )

    # Lags 1 a 4
    for i in range(1, 5):

        coluna_lag = f"brent_lag{i}"

        df_valido = df_comb.filter(
            col(coluna_lag).isNotNull() &
            col("var_ppi_pct").isNotNull()
        )

        r = (
            df_valido
            .select(corr(coluna_lag, "var_ppi_pct"))
            .first()[0]
        )

        resultados_corrigidos.append(
            (
                combustivel,
                "Brent → PPI | 2018–2026",
                i,
                r,
                df_valido.count()
            )
        )

correlacoes_brent_ppi_corrigidas = spark.createDataFrame(
    resultados_corrigidos,
    [
        "combustivel",
        "relacao",
        "lag_semanas",
        "correlacao",
        "n"
    ]
)

display(
    correlacoes_brent_ppi_corrigidas
    .orderBy("combustivel", "lag_semanas")
)

combustivel,relacao,lag_semanas,correlacao,n
DIESEL,Brent → PPI | 2018–2026,0,0.5884883182627068,404
DIESEL,Brent → PPI | 2018–2026,1,0.2730166607329578,403
DIESEL,Brent → PPI | 2018–2026,2,0.06910410697219892,403
DIESEL,Brent → PPI | 2018–2026,3,0.0019309998537232135,403
DIESEL,Brent → PPI | 2018–2026,4,0.0023818686137066745,402
GASOLINA,Brent → PPI | 2018–2026,0,0.6903602071689033,408
GASOLINA,Brent → PPI | 2018–2026,1,0.3248773696609679,407
GASOLINA,Brent → PPI | 2018–2026,2,-0.017487095292935868,406
GASOLINA,Brent → PPI | 2018–2026,3,-0.018928490041430168,405
GASOLINA,Brent → PPI | 2018–2026,4,-0.05639041079288966,404


In [0]:
(
    correlacoes_brent_ppi_corrigidas
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.correlacao_brent_ppi")
)

print("Tabela workspace.gold.correlacao_brent_ppi criada.")

Tabela workspace.gold.correlacao_brent_ppi criada.


In [0]:
%whos

Variable                           Type            Data/Info
------------------------------------------------------------
Window                             type            <class 'pyspark.sql.window.Window'>
anp_gold_semanal                   DataFrame       DataFrame[semana: date, p<...>, ufs_observadas: bigint]
anp_integracao                     DataFrame       DataFrame[semana: date, c<...>, ufs_observadas: bigint]
anp_ppi                            DataFrame       DataFrame[semana: date, c<...>dades_observadas: bigint]
anp_silver                         DataFrame       DataFrame[data_coleta: da<...>timestamp, fonte: string]
avg                                function        <function avg at 0xff7734591620>
base_integrada                     DataFrame       DataFrame[semana: date, c<...> dias_observados: bigint]
base_lags                          DataFrame       DataFrame[semana: date, c<...>double, ppi_lag4: double]
base_variacoes                     DataFrame       DataFrame[sema

In [0]:
display(
    spark.sql("""
        SHOW TABLES IN workspace.gold
    """)
)

database,tableName,isTemporary
gold,anp_semanal,false
gold,base_integrada,false
gold,brent_semanal,false
gold,correlacao_brent_ppi,false
gold,correlacoes_anp,false
gold,ppi_semanal,false


# Reconstrução da camada Gold a partir das tabelas Silver persistidas

Nesta etapa, as tabelas da camada Silver são utilizadas como fonte para
reconstrução e persistência das tabelas analíticas da camada Gold.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.functions import (
    col,
    avg,
    min,
    max,
    count,
    countDistinct,
    lag,
    lit,
    when,
    expr,
    date_trunc
)
from pyspark.sql.window import Window

# Carregamento das tabelas persistidas na camada Silver
brent_silver = spark.table("workspace.silver.brent")
ppi_silver = spark.table("workspace.silver.ppi")
anp_silver = spark.table("workspace.silver.precos_anp")

print("Tabelas Silver carregadas com sucesso.")
print(f"Brent: {brent_silver.count():,}")
print(f"PPI: {ppi_silver.count():,}")
print(f"ANP: {anp_silver.count():,}")

Tabelas Silver carregadas com sucesso.
Brent: 9,973
PPI: 12,142
ANP: 1,374,810


## Construção e persistência das tabelas analíticas da camada Gold

A partir das tabelas tratadas e persistidas na camada Silver,
são construídas séries temporais semanais para Brent, PPI e preços
de combustíveis da ANP.

A camada Gold concentra os dados agregados e integrados utilizados
nas análises estatísticas do projeto, incluindo variações semanais,
defasagens temporais (lags) e correlações entre os indicadores.

In [0]:
(
    brent_gold_semanal
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.brent_semanal")
)

In [0]:
(
    ppi_gold_semanal
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.ppi_semanal")
)

print("Tabela workspace.gold.ppi_semanal criada com sucesso.")

Tabela workspace.gold.ppi_semanal criada com sucesso.


In [0]:
(
    anp_gold_semanal
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.anp_semanal")
)

print("Tabela workspace.gold.anp_semanal criada com sucesso.")

Tabela workspace.gold.anp_semanal criada com sucesso.


In [0]:
print("Registros base_integrada:", base_integrada.count())

base_integrada.printSchema()

display(
    base_integrada
    .orderBy(col("semana").desc(), "combustivel")
    .limit(10)
)

Registros base_integrada: 176
root
 |-- semana: date (nullable = true)
 |-- combustivel: string (nullable = true)
 |-- preco_medio_anp: double (nullable = true)
 |-- preco_min_anp: double (nullable = true)
 |-- preco_max_anp: double (nullable = true)
 |-- observacoes: long (nullable = false)
 |-- ufs_observadas: long (nullable = false)
 |-- ppi_medio: double (nullable = true)
 |-- ppi_min: double (nullable = true)
 |-- ppi_max: double (nullable = true)
 |-- localidades_observadas: long (nullable = true)
 |-- brent_medio_usd: double (nullable = true)
 |-- brent_min_usd: double (nullable = true)
 |-- brent_max_usd: double (nullable = true)
 |-- dias_observados: long (nullable = true)



semana,combustivel,preco_medio_anp,preco_min_anp,preco_max_anp,observacoes,ufs_observadas,ppi_medio,ppi_min,ppi_max,localidades_observadas,brent_medio_usd,brent_min_usd,brent_max_usd,dias_observados
2026-08-31,DIESEL,6.937380636604777,5.63,8.99,1508,26,6.290048625,6.188046,6.436979999999999,16,99.0925,96.02,102.24,4
2026-08-31,GASOLINA,6.556554021894337,5.44,8.99,2101,26,3.6398353750000005,3.56902,3.7864180000000003,16,99.0925,96.02,102.24,4
2026-08-24,DIESEL,6.914838396897216,5.74,9.27,3094,27,5.947009249999999,5.84642,6.094814,16,89.72999999999999,87.77,92.71,5
2026-08-24,GASOLINA,6.548775510204077,5.39,8.99,4361,27,3.5324707500000003,3.4615859999999996,3.6801239999999997,16,89.72999999999999,87.77,92.71,5
2026-08-17,DIESEL,6.923696816892528,5.77,9.19,3173,27,6.109502750000001,6.007024,6.2577359999999995,16,94.20200000000001,92.37,96.92,5
2026-08-17,GASOLINA,6.560445289773992,5.49,8.99,4469,27,3.5684145000000003,3.497044,3.716348,16,94.20200000000001,92.37,96.92,5
2026-08-10,DIESEL,6.954237018158645,5.77,9.27,3139,27,5.7894380000000005,5.69397,5.937270000000001,16,92.51399999999998,92.02,93.26,5
2026-08-10,GASOLINA,6.57642552235452,5.49,8.99,4451,27,3.4824391249999995,3.4119699999999997,3.630764,16,92.51399999999998,92.02,93.26,5
2026-08-03,DIESEL,6.976342637151106,5.84,9.19,3117,27,5.38436225,5.285626000000001,5.530756,16,87.85799999999999,86.47,89.65,5
2026-08-03,GASOLINA,6.59284098620221,5.49,8.99,4421,27,3.3443706250000003,3.27404,3.490774,16,87.85799999999999,86.47,89.65,5


In [0]:
(
    base_integrada
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.base_integrada")
)

print("Tabela workspace.gold.base_integrada criada com sucesso.")

Tabela workspace.gold.base_integrada criada com sucesso.


In [0]:
display(
    spark.sql("""
        SHOW TABLES IN workspace.gold
    """)
)

database,tableName,isTemporary
gold,anp_semanal,false
gold,base_integrada,false
gold,brent_semanal,false
gold,correlacao_brent_ppi,false
gold,correlacoes_anp,false
gold,ppi_semanal,false


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, lag

janela_integrada = (
    Window
    .partitionBy("combustivel")
    .orderBy("semana")
)

base_integrada_variacoes = (
    base_integrada

    # Valores da semana anterior
    .withColumn(
        "brent_anterior",
        lag("brent_medio_usd", 1).over(janela_integrada)
    )
    .withColumn(
        "ppi_anterior",
        lag("ppi_medio", 1).over(janela_integrada)
    )
    .withColumn(
        "anp_anterior",
        lag("preco_medio_anp", 1).over(janela_integrada)
    )

    # Variações percentuais semanais
    .withColumn(
        "var_brent_pct",
        ((col("brent_medio_usd") - col("brent_anterior"))
         / col("brent_anterior")) * 100
    )
    .withColumn(
        "var_ppi_pct",
        ((col("ppi_medio") - col("ppi_anterior"))
         / col("ppi_anterior")) * 100
    )
    .withColumn(
        "var_anp_pct",
        ((col("preco_medio_anp") - col("anp_anterior"))
         / col("anp_anterior")) * 100
    )
)

display(
    base_integrada_variacoes
    .select(
        "semana",
        "combustivel",
        "var_brent_pct",
        "var_ppi_pct",
        "var_anp_pct"
    )
    .orderBy(col("semana").desc(), "combustivel")
    .limit(20)
)

semana,combustivel,var_brent_pct,var_ppi_pct,var_anp_pct
2026-08-31,DIESEL,10.434080017831285,5.768267049525776,0.3259980698556697
2026-08-31,GASOLINA,10.434080017831285,3.0393634540356707,0.11877810864167106
2026-08-24,DIESEL,-4.7472452814165536,-2.6596845381565495,-0.1279434993990541
2026-08-24,GASOLINA,-4.7472452814165536,-1.007275079730774,-0.17788090677482848
2026-08-17,DIESEL,1.8245887108978436,5.528425211566292,-0.4391596257989909
2026-08-17,GASOLINA,1.8245887108978436,2.4688263574456553,-0.2429926793241595
2026-08-10,DIESEL,5.299460493068351,7.523189027632786,-0.31686544285720264
2026-08-10,GASOLINA,5.299460493068351,4.128385142720243,-0.2489892275884365
2026-08-03,DIESEL,-4.114462827956515,-7.003751357732267,-0.162649783357123
2026-08-03,GASOLINA,-4.114462827956515,-8.798124051143878,-0.1906256807865855


In [0]:
from pyspark.sql.functions import lag
from pyspark.sql.window import Window

janela_lags = (
    Window
    .partitionBy("combustivel")
    .orderBy("semana")
)

base_integrada_lags = base_integrada_variacoes

for i in range(1, 5):

    # Lags do Brent
    base_integrada_lags = (
        base_integrada_lags
        .withColumn(
            f"brent_lag{i}",
            lag("var_brent_pct", i).over(janela_lags)
        )
    )

    # Lags do PPI
    base_integrada_lags = (
        base_integrada_lags
        .withColumn(
            f"ppi_lag{i}",
            lag("var_ppi_pct", i).over(janela_lags)
        )
    )

display(
    base_integrada_lags
    .select(
        "semana",
        "combustivel",
        "var_brent_pct",
        "brent_lag1",
        "brent_lag2",
        "brent_lag3",
        "brent_lag4",
        "var_ppi_pct",
        "ppi_lag1",
        "ppi_lag2",
        "ppi_lag3",
        "ppi_lag4",
        "var_anp_pct"
    )
    .orderBy(col("semana").desc(), "combustivel")
    .limit(20)
)

semana,combustivel,var_brent_pct,brent_lag1,brent_lag2,brent_lag3,brent_lag4,var_ppi_pct,ppi_lag1,ppi_lag2,ppi_lag3,ppi_lag4,var_anp_pct
2026-08-31,DIESEL,10.434080017831285,-4.7472452814165536,1.8245887108978436,5.299460493068351,-4.114462827956515,5.768267049525776,-2.6596845381565495,5.528425211566292,7.523189027632786,-7.003751357732267,0.3259980698556697
2026-08-31,GASOLINA,10.434080017831285,-4.7472452814165536,1.8245887108978436,5.299460493068351,-4.114462827956515,3.0393634540356707,-1.007275079730774,2.4688263574456553,4.128385142720243,-8.798124051143878,0.11877810864167106
2026-08-24,DIESEL,-4.7472452814165536,1.8245887108978436,5.299460493068351,-4.114462827956515,-4.671341476102301,-2.6596845381565495,5.528425211566292,7.523189027632786,-7.003751357732267,3.217459633092038,-0.1279434993990541
2026-08-24,GASOLINA,-4.7472452814165536,1.8245887108978436,5.299460493068351,-4.114462827956515,-4.671341476102301,-1.007275079730774,2.4688263574456553,4.128385142720243,-8.798124051143878,-2.061856977968688,-0.17788090677482848
2026-08-17,DIESEL,1.8245887108978436,5.299460493068351,-4.114462827956515,-4.671341476102301,15.908159081590806,5.528425211566292,7.523189027632786,-7.003751357732267,3.217459633092038,6.25478204392674,-0.4391596257989909
2026-08-17,GASOLINA,1.8245887108978436,5.299460493068351,-4.114462827956515,-4.671341476102301,15.908159081590806,2.4688263574456553,4.128385142720243,-8.798124051143878,-2.061856977968688,3.9081875031676123,-0.2429926793241595
2026-08-10,DIESEL,5.299460493068351,-4.114462827956515,-4.671341476102301,15.908159081590806,13.089133755182193,7.523189027632786,-7.003751357732267,3.217459633092038,6.25478204392674,13.084999890887264,-0.31686544285720264
2026-08-10,GASOLINA,5.299460493068351,-4.114462827956515,-4.671341476102301,15.908159081590806,13.089133755182193,4.128385142720243,-8.798124051143878,-2.061856977968688,3.9081875031676123,8.652411639868058,-0.2489892275884365
2026-08-03,DIESEL,-4.114462827956515,-4.671341476102301,15.908159081590806,13.089133755182193,5.2051649928263775,-7.003751357732267,3.217459633092038,6.25478204392674,13.084999890887264,5.692485503459753,-0.162649783357123
2026-08-03,GASOLINA,-4.114462827956515,-4.671341476102301,15.908159081590806,13.089133755182193,5.2051649928263775,-8.798124051143878,-2.061856977968688,3.9081875031676123,8.652411639868058,3.5096428332606795,-0.1906256807865855


In [0]:
from pyspark.sql.functions import col, corr

def calcular_correlacoes_gold(
    df,
    x_base,
    x_prefixo,
    y,
    relacao
):
    resultados = []

    for combustivel in ["DIESEL", "GASOLINA"]:

        dados = df.filter(
            col("combustivel") == combustivel
        )

        for lag_num in range(0, 5):

            coluna_x = (
                x_base
                if lag_num == 0
                else f"{x_prefixo}{lag_num}"
            )

            dados_validos = (
                dados
                .select(coluna_x, y)
                .dropna()
            )

            n = dados_validos.count()

            r = (
                dados_validos
                .select(
                    corr(
                        col(coluna_x),
                        col(y)
                    ).alias("correlacao")
                )
                .first()["correlacao"]
            )

            resultados.append(
                (
                    combustivel,
                    relacao,
                    lag_num,
                    r,
                    n
                )
            )

    return spark.createDataFrame(
        resultados,
        [
            "combustivel",
            "relacao",
            "lag_semanas",
            "correlacao",
            "n"
        ]
    )


# PPI → preço ANP
correlacoes_ppi_anp_final = calcular_correlacoes_gold(
    base_integrada_lags,
    "var_ppi_pct",
    "ppi_lag",
    "var_anp_pct",
    "PPI → ANP | janela comum"
)


# Brent → preço ANP
correlacoes_brent_anp_final = calcular_correlacoes_gold(
    base_integrada_lags,
    "var_brent_pct",
    "brent_lag",
    "var_anp_pct",
    "Brent → ANP | janela comum"
)


display(
    correlacoes_ppi_anp_final
    .unionByName(correlacoes_brent_anp_final)
    .orderBy(
        "combustivel",
        "relacao",
        "lag_semanas"
    )
)

combustivel,relacao,lag_semanas,correlacao,n
DIESEL,Brent → ANP | janela comum,0,0.32147463430314166,87
DIESEL,Brent → ANP | janela comum,1,0.47000729303341204,86
DIESEL,Brent → ANP | janela comum,2,0.23561794946380354,85
DIESEL,Brent → ANP | janela comum,3,0.11496080278812089,84
DIESEL,Brent → ANP | janela comum,4,0.027313726327869603,83
DIESEL,PPI → ANP | janela comum,0,0.340425365692268,87
DIESEL,PPI → ANP | janela comum,1,0.6555478676836616,86
DIESEL,PPI → ANP | janela comum,2,0.41121891326460763,85
DIESEL,PPI → ANP | janela comum,3,0.22967983645353884,84
DIESEL,PPI → ANP | janela comum,4,-0.020017000281071513,83


In [0]:
correlacoes_anp_final = (
    correlacoes_ppi_anp_final
    .unionByName(correlacoes_brent_anp_final)
)

(
    correlacoes_anp_final
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable("workspace.gold.correlacoes_anp")
)

print("Tabela workspace.gold.correlacoes_anp criada com sucesso.")

Tabela workspace.gold.correlacoes_anp criada com sucesso.


In [0]:
# Validação final da camada Gold

print("TABELAS DA CAMADA GOLD")

display(
    spark.sql("""
        SHOW TABLES IN workspace.gold
    """)
)

print("\nCONTAGEM DE REGISTROS")

tabelas_gold = [
    "brent_semanal",
    "ppi_semanal",
    "anp_semanal",
    "base_integrada",
    "correlacao_brent_ppi",
    "correlacoes_anp"
]

for tabela in tabelas_gold:
    df = spark.table(f"workspace.gold.{tabela}")
    print(f"{tabela}: {df.count():,} registros")

TABELAS DA CAMADA GOLD


database,tableName,isTemporary
gold,anp_semanal,false
gold,base_integrada,false
gold,brent_semanal,false
gold,correlacao_brent_ppi,false
gold,correlacoes_anp,false
gold,ppi_semanal,false



CONTAGEM DE REGISTROS
brent_semanal: 2,052 registros
ppi_semanal: 818 registros
anp_semanal: 176 registros
base_integrada: 176 registros
correlacao_brent_ppi: 10 registros
correlacoes_anp: 20 registros


## Análise exploratória e interpretação dos resultados

Com as camadas Bronze, Silver e Gold estruturadas e persistidas, esta etapa utiliza as tabelas analíticas da camada Gold para investigar a relação temporal entre o preço internacional do petróleo Brent, os preços de referência do PPI e os preços de combustíveis observados ao consumidor pela ANP.

A análise considera variações percentuais semanais e defasagens temporais de até quatro semanas, buscando identificar padrões de associação entre os indicadores. As correlações observadas são interpretadas como medidas de associação estatística, não como evidência de causalidade.

In [0]:
from pyspark.sql.functions import min, max, count

# Brent
print("\nBrent")
display(
    spark.table("workspace.gold.brent_semanal")
    .agg(
        min("semana").alias("inicio"),
        max("semana").alias("fim"),
        count("*").alias("registros")
    )
)

# PPI
print("\nPPI")
display(
    spark.table("workspace.gold.ppi_semanal")
    .agg(
        min("data_inicio").alias("inicio"),
        max("data_fim").alias("fim"),
        count("*").alias("registros")
    )
)

# ANP
print("\nANP")
display(
    spark.table("workspace.gold.anp_semanal")
    .agg(
        min("semana").alias("inicio"),
        max("semana").alias("fim"),
        count("*").alias("registros")
    )
)

# Base integrada
print("\nBase integrada")
display(
    spark.table("workspace.gold.base_integrada")
    .agg(
        min("semana").alias("inicio"),
        max("semana").alias("fim"),
        count("*").alias("registros")
    )
)


Brent


inicio,fim,registros
1987-05-18,2026-09-07,2052



PPI


inicio,fim,registros
2018-11-05,2026-09-04,818



ANP


inicio,fim,registros
2024-12-30,2026-08-31,176



Base integrada


inicio,fim,registros
2024-12-30,2026-08-31,176


## Estatísticas descritivas

Inicialmente, são analisadas estatísticas descritivas das séries que compõem a camada Gold, permitindo caracterizar seus níveis, dispersão e amplitude antes da análise de suas relações temporais.

In [0]:
from pyspark.sql.functions import (
    avg,
    min,
    max,
    stddev,
    round as spark_round
)

# Brent
estatisticas_brent = (
    spark.table("workspace.gold.brent_semanal")
    .agg(
        spark_round(avg("brent_medio_usd"), 2).alias("media"),
        spark_round(stddev("brent_medio_usd"), 2).alias("desvio_padrao"),
        spark_round(min("brent_medio_usd"), 2).alias("minimo"),
        spark_round(max("brent_medio_usd"), 2).alias("maximo")
    )
)

print("=== Brent (US$/barril) ===")
display(estatisticas_brent)


# PPI por combustível
estatisticas_ppi = (
    spark.table("workspace.gold.ppi_semanal")
    .groupBy("produto")
    .agg(
        spark_round(avg("ppi_medio"), 2).alias("media"),
        spark_round(stddev("ppi_medio"), 2).alias("desvio_padrao"),
        spark_round(min("ppi_medio"), 2).alias("minimo"),
        spark_round(max("ppi_medio"), 2).alias("maximo")
    )
    .orderBy("produto")
)

print("=== PPI ===")
display(estatisticas_ppi)


# ANP por combustível
estatisticas_anp = (
    spark.table("workspace.gold.anp_semanal")
    .groupBy("produto")
    .agg(
        spark_round(avg("preco_medio_anp"), 2).alias("media"),
        spark_round(stddev("preco_medio_anp"), 2).alias("desvio_padrao"),
        spark_round(min("preco_medio_anp"), 2).alias("minimo"),
        spark_round(max("preco_medio_anp"), 2).alias("maximo")
    )
    .orderBy("produto")
)

print("=== Preços ANP (R$/litro) ===")
display(estatisticas_anp)

=== Brent (US$/barril) ===


media,desvio_padrao,minimo,maximo
51.58,32.9,9.44,141.07


=== PPI ===


produto,media,desvio_padrao,minimo,maximo
DIESEL,3.39,1.17,1.34,6.4
GASOLINA,2.71,0.8,0.67,4.95


=== Preços ANP (R$/litro) ===


produto,media,desvio_padrao,minimo,maximo
DIESEL S10,6.48,0.47,6.09,7.58
GASOLINA,6.37,0.2,6.15,6.81


## Evolução histórica do preço do petróleo Brent

A série histórica do Brent permite observar a evolução do preço internacional do petróleo ao longo do período disponível na base. A visualização auxilia na identificação de períodos de maior volatilidade e fornece contexto para as análises posteriores envolvendo PPI e preços de combustíveis no mercado brasileiro.

In [0]:
brent_grafico = (
    spark.table("workspace.gold.brent_semanal")
    .select(
        "semana",
        "brent_medio_usd"
    )
    .orderBy("semana")
)

display(brent_grafico)

semana,brent_medio_usd
1987-05-18,18.543333333333333
1987-05-25,18.602
1987-06-01,18.701999999999998
1987-06-08,18.754
1987-06-15,19.0075
1987-06-22,18.906
1987-06-29,19.157999999999998
1987-07-06,19.574000000000005
1987-07-13,20.203999999999997
1987-07-20,20.192


Databricks visualization. Run in Databricks to view.

## Evolução do PPI de combustíveis

A série do PPI permite acompanhar a evolução dos preços de referência de gasolina e diesel ao longo do período disponível. A análise separada por produto possibilita observar diferenças de nível e comportamento entre os dois combustíveis e fornece a base para a posterior comparação com as variações do petróleo Brent.

In [0]:
ppi_grafico = (
    spark.table("workspace.gold.ppi_semanal")
    .select(
        "data_inicio",
        "produto",
        "ppi_medio"
    )
    .orderBy("data_inicio", "produto")
)

display(ppi_grafico)

data_inicio,produto,ppi_medio
2018-11-05,DIESEL,2.269182
2018-11-05,GASOLINA,1.6240896
2018-11-12,DIESEL,2.1791376
2018-11-12,GASOLINA,1.5455747999999998
2018-11-17,DIESEL,2.1250366666666665
2018-11-19,GASOLINA,1.5052953333333334
2018-11-26,DIESEL,2.0116776
2018-11-26,GASOLINA,1.426098
2018-12-03,DIESEL,2.0175799999999997
2018-12-03,GASOLINA,1.43934


Databricks visualization. Run in Databricks to view.

## Evolução dos preços de combustíveis ao consumidor

Os dados da ANP representam os preços observados nos postos de combustíveis, permitindo analisar a evolução semanal dos preços médios de gasolina e diesel S10 no mercado brasileiro. Essa série corresponde à etapa final da cadeia analisada no projeto, possibilitando posteriormente avaliar sua relação com as variações do PPI e do petróleo Brent.

In [0]:
anp_grafico = (
    spark.table("workspace.gold.anp_semanal")
    .select(
        "semana",
        "produto",
        "preco_medio_anp"
    )
    .orderBy("semana", "produto")
)

display(anp_grafico)

semana,produto,preco_medio_anp
2024-12-30,DIESEL S10,6.171021126760562
2024-12-30,GASOLINA,6.187310996563575
2025-01-06,DIESEL S10,6.162312565997889
2025-01-06,GASOLINA,6.150376993716766
2025-01-13,DIESEL S10,6.176929300035176
2025-01-13,GASOLINA,6.1622678658240115
2025-01-20,DIESEL S10,6.200104694360018
2025-01-20,GASOLINA,6.204424923869755
2025-01-27,DIESEL S10,6.2077021276595765
2025-01-27,GASOLINA,6.21676971762415


Databricks visualization. Run in Databricks to view.

## Relação entre as variações de Brent, PPI e preços ao consumidor

Para tornar comparáveis séries expressas em unidades distintas, a análise a seguir utiliza as variações percentuais semanais dos indicadores.

A comparação permite observar como oscilações no preço internacional do petróleo Brent se relacionam com alterações no PPI e, posteriormente, com os preços de combustíveis observados pela ANP. Além da relação contemporânea, são consideradas defasagens temporais de até quatro semanas para investigar como as variações de uma série podem se refletir nas demais ao longo das semanas seguintes.

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col, lag

base_analise = spark.table("workspace.gold.base_integrada")

janela_analise = (
    Window
    .partitionBy("combustivel")
    .orderBy("semana")
)

variacoes_grafico = (
    base_analise

    .withColumn(
        "brent_anterior",
        lag("brent_medio_usd").over(janela_analise)
    )
    .withColumn(
        "ppi_anterior",
        lag("ppi_medio").over(janela_analise)
    )
    .withColumn(
        "anp_anterior",
        lag("preco_medio_anp").over(janela_analise)
    )

    .withColumn(
        "var_brent_pct",
        ((col("brent_medio_usd") / col("brent_anterior")) - 1) * 100
    )
    .withColumn(
        "var_ppi_pct",
        ((col("ppi_medio") / col("ppi_anterior")) - 1) * 100
    )
    .withColumn(
        "var_anp_pct",
        ((col("preco_medio_anp") / col("anp_anterior")) - 1) * 100
    )

    .select(
        "semana",
        "combustivel",
        "var_brent_pct",
        "var_ppi_pct",
        "var_anp_pct"
    )
)

display(
    variacoes_grafico
    .orderBy(col("semana").desc(), "combustivel")
)

semana,combustivel,var_brent_pct,var_ppi_pct,var_anp_pct
2026-08-31,DIESEL,10.434080017831281,5.768267049525799,0.3259980698562037
2026-08-31,GASOLINA,10.434080017831281,3.0393634540356818,0.11877810864244776
2026-08-24,DIESEL,-4.747245281416557,-2.6596845381565926,-0.12794349939904137
2026-08-24,GASOLINA,-4.747245281416557,-1.007275079730785,-0.17788090677479662
2026-08-17,DIESEL,1.8245887108978343,5.528425211566312,-0.43915962579893497
2026-08-17,GASOLINA,1.8245887108978343,2.468826357445675,-0.24299267932417212
2026-08-10,DIESEL,5.299460493068353,7.5231890276327595,-0.316865442857428
2026-08-10,GASOLINA,5.299460493068353,4.128385142720226,-0.24898922758861408
2026-08-03,DIESEL,-4.114462827956511,-7.003751357732247,-0.1626497833570606
2026-08-03,GASOLINA,-4.114462827956511,-8.798124051143896,-0.19062568078642705


In [0]:
from pyspark.sql.functions import col, lag
from pyspark.sql.window import Window

# Janela temporal por combustível
w = (
    Window
    .partitionBy("combustivel")
    .orderBy("semana")
)

variacoes_grafico = (
    spark.table("workspace.gold.base_integrada")
    
    .withColumn(
        "var_brent_pct",
        (
            (col("brent_medio_usd") - lag("brent_medio_usd").over(w))
            / lag("brent_medio_usd").over(w)
        ) * 100
    )
    
    .withColumn(
        "var_ppi_pct",
        (
            (col("ppi_medio") - lag("ppi_medio").over(w))
            / lag("ppi_medio").over(w)
        ) * 100
    )
    
    .withColumn(
        "var_anp_pct",
        (
            (col("preco_medio_anp") - lag("preco_medio_anp").over(w))
            / lag("preco_medio_anp").over(w)
        ) * 100
    )
    
    .select(
        "semana",
        "combustivel",
        "var_brent_pct",
        "var_ppi_pct",
        "var_anp_pct"
    )
    
    .orderBy("semana")
)

display(variacoes_grafico)

semana,combustivel,var_brent_pct,var_ppi_pct,var_anp_pct
2024-12-30,GASOLINA,null,null,null
2024-12-30,DIESEL,null,null,null
2025-01-06,GASOLINA,3.6011667992575025,-0.7797753051843866,-0.5969314111949713
2025-01-06,DIESEL,3.6011667992575025,-0.6004500596171819,-0.14112025520231247
2025-01-13,DIESEL,5.961400634790624,6.557793361069628,0.23719559630809134
2025-01-13,GASOLINA,5.961400634790624,2.4176891278900015,0.19333566250317658
2025-01-20,DIESEL,-3.476097301736841,-3.1055000949654765,0.3751928053427826
2025-01-20,GASOLINA,-3.476097301736841,-3.535966522876357,0.6841159612607424
2025-01-27,DIESEL,-3.1833425096351133,-5.448011750008954,0.12253717758136577
2025-01-27,GASOLINA,-3.1833425096351133,-3.0222111152736484,0.19896757404382917


In [0]:
from pyspark.sql.functions import col

variacoes_diesel_grafico = (
    variacoes_grafico
    .filter(col("combustivel") == "DIESEL")
    .select(
        col("semana"),
        col("var_brent_pct").alias("Brent (%)"),
        col("var_ppi_pct").alias("PPI Diesel (%)"),
        col("var_anp_pct").alias("Preço consumidor (%)")
    )
    .orderBy("semana")
)

display(variacoes_diesel_grafico)

semana,Brent (%),PPI Diesel (%),Preço consumidor (%)
2024-12-30,null,null,null
2025-01-06,3.6011667992575025,-0.6004500596171819,-0.14112025520231247
2025-01-13,5.961400634790624,6.557793361069628,0.23719559630809134
2025-01-20,-3.476097301736841,-3.1055000949654765,0.3751928053427826
2025-01-27,-3.1833425096351133,-5.448011750008954,0.12253717758136577
2025-02-03,-2.5487256371814015,-1.5894375871926931,4.011115150318225
2025-02-10,0.8116710875331495,0.30928178388059,0.4904767059247579
2025-02-17,0.09998421301898833,-1.3788354246834902,0.051241806720303276
2025-02-24,-2.4129954789191443,-0.5697949772863138,-0.1803388977490608
2025-03-03,-3.1271884932392435,-2.1914524797220993,-0.19859926141597467


Databricks visualization. Run in Databricks to view.

In [0]:
variacoes_gasolina_grafico = (
    variacoes_grafico
    .filter(col("combustivel") == "GASOLINA")
    .select(
        col("semana"),
        col("var_brent_pct").alias("Brent (%)"),
        col("var_ppi_pct").alias("PPI Gasolina (%)"),
        col("var_anp_pct").alias("Preço consumidor (%)")
    )
    .orderBy("semana")
)

display(variacoes_gasolina_grafico)

semana,Brent (%),PPI Gasolina (%),Preço consumidor (%)
2024-12-30,null,null,null
2025-01-06,3.6011667992575025,-0.7797753051843866,-0.5969314111949713
2025-01-13,5.961400634790624,2.4176891278900015,0.19333566250317658
2025-01-20,-3.476097301736841,-3.535966522876357,0.6841159612607424
2025-01-27,-3.1833425096351133,-3.0222111152736484,0.19896757404382917
2025-02-03,-2.5487256371814015,0.4484232564139007,2.248927874651898
2025-02-10,0.8116710875331495,0.6445336815018774,0.2985450097821002
2025-02-17,0.09998421301898833,-0.6599728427919125,-0.06677990187648133
2025-02-24,-2.4129954789191443,-0.7689821332347262,-0.09635722532184032
2025-03-03,-3.1271884932392435,0.05589668550284532,-0.014388898248668196


Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import col

corr_brent_ppi_grafico = (
    spark.table("workspace.gold.correlacao_brent_ppi")
    .select(
        "combustivel",
        "relacao",
        "lag_semanas",
        "correlacao",
        "n"
    )
)

corr_anp_grafico = (
    spark.table("workspace.gold.correlacoes_anp")
    .select(
        "combustivel",
        "relacao",
        "lag_semanas",
        "correlacao",
        "n"
    )
)

correlacoes_grafico = (
    corr_brent_ppi_grafico
    .unionByName(corr_anp_grafico)
    .orderBy(
        "combustivel",
        "relacao",
        "lag_semanas"
    )
)

display(correlacoes_grafico)

combustivel,relacao,lag_semanas,correlacao,n
DIESEL,Brent → ANP | janela comum,0,0.32147463430314166,87
DIESEL,Brent → ANP | janela comum,1,0.47000729303341204,86
DIESEL,Brent → ANP | janela comum,2,0.23561794946380354,85
DIESEL,Brent → ANP | janela comum,3,0.11496080278812089,84
DIESEL,Brent → ANP | janela comum,4,0.027313726327869603,83
DIESEL,Brent → PPI | 2018–2026,0,0.5884883182627068,404
DIESEL,Brent → PPI | 2018–2026,1,0.2730166607329578,403
DIESEL,Brent → PPI | 2018–2026,2,0.06910410697219892,403
DIESEL,Brent → PPI | 2018–2026,3,0.0019309998537232135,403
DIESEL,Brent → PPI | 2018–2026,4,0.0023818686137066745,402


In [0]:
brent_ppi_lags_grafico = (
    correlacoes_grafico
    .filter(col("relacao") == "Brent → PPI | 2018–2026")
    .select(
        "lag_semanas",
        "combustivel",
        "correlacao"
    )
    .orderBy("lag_semanas", "combustivel")
)

display(brent_ppi_lags_grafico)

lag_semanas,combustivel,correlacao
0,DIESEL,0.5884883182627068
0,GASOLINA,0.6903602071689033
1,DIESEL,0.2730166607329578
1,GASOLINA,0.3248773696609679
2,DIESEL,0.06910410697219892
2,GASOLINA,-0.017487095292935868
3,DIESEL,0.0019309998537232135
3,GASOLINA,-0.018928490041430168
4,DIESEL,0.0023818686137066745
4,GASOLINA,-0.05639041079288966


Databricks visualization. Run in Databricks to view.

In [0]:
from pyspark.sql.functions import col, when

anp_lags_grafico = (
    correlacoes_grafico
    .filter(
        col("relacao").isin(
            "Brent → ANP | janela comum",
            "PPI → ANP | janela comum"
        )
    )
    .withColumn(
        "indicador",
        when(
            col("relacao") == "Brent → ANP | janela comum",
            "Brent"
        ).otherwise("PPI")
    )
    .select(
        "lag_semanas",
        "combustivel",
        "indicador",
        "correlacao"
    )
    .orderBy("combustivel", "indicador", "lag_semanas")
)

display(anp_lags_grafico)

lag_semanas,combustivel,indicador,correlacao
0,DIESEL,Brent,0.32147463430314166
1,DIESEL,Brent,0.47000729303341204
2,DIESEL,Brent,0.23561794946380354
3,DIESEL,Brent,0.11496080278812089
4,DIESEL,Brent,0.027313726327869603
0,DIESEL,PPI,0.340425365692268
1,DIESEL,PPI,0.6555478676836616
2,DIESEL,PPI,0.41121891326460763
3,DIESEL,PPI,0.22967983645353884
4,DIESEL,PPI,-0.020017000281071513


In [0]:
anp_lags_diesel_grafico = (
    anp_lags_grafico
    .filter(col("combustivel") == "DIESEL")
    .select(
        "lag_semanas",
        "indicador",
        "correlacao"
    )
    .orderBy("lag_semanas", "indicador")
)

display(anp_lags_diesel_grafico)

lag_semanas,indicador,correlacao
0,Brent,0.32147463430314166
0,PPI,0.340425365692268
1,Brent,0.47000729303341204
1,PPI,0.6555478676836616
2,Brent,0.23561794946380354
2,PPI,0.41121891326460763
3,Brent,0.11496080278812089
3,PPI,0.22967983645353884
4,Brent,0.027313726327869603
4,PPI,-0.020017000281071513


Databricks visualization. Run in Databricks to view.

In [0]:
anp_lags_gasolina_grafico = (
    anp_lags_grafico
    .filter(col("combustivel") == "GASOLINA")
    .select(
        "lag_semanas",
        "indicador",
        "correlacao"
    )
    .orderBy("lag_semanas", "indicador")
)

display(anp_lags_gasolina_grafico)

lag_semanas,indicador,correlacao
0,Brent,0.18668524845188234
0,PPI,0.3151432140655186
1,Brent,0.40792614657465526
1,PPI,0.47894709882996434
2,Brent,0.30295808029482696
2,PPI,0.35333371278991654
3,Brent,0.1513556085819195
3,PPI,0.10813589345402648
4,Brent,-0.008470543869503657
4,PPI,-0.07047963433644303


Databricks visualization. Run in Databricks to view.

## Interpretação das correlações com os preços 
##ao consumidor

As correlações entre as variações do Brent e do PPI e as variações dos preços dos combustíveis ao consumidor apresentam comportamento semelhante para diesel e gasolina. Em ambos os casos, as maiores correlações são observadas com defasagem de uma semana (lag 1).

Para o diesel, a correlação entre a variação do PPI e a variação do preço ao consumidor aumenta de 0,340 no lag 0 para 0,656 no lag 1, enquanto a relação com o Brent passa de 0,321 para 0,470. Para a gasolina, o PPI apresenta correlação de 0,315 no lag 0 e 0,479 no lag 1, enquanto o Brent passa de 0,187 para 0,408.

A partir do lag 2, as correlações diminuem e se aproximam de zero no lag 4. Os resultados indicam, portanto, que as associações mais fortes entre as variações dos indicadores upstream e os preços ao consumidor, na janela analisada, ocorrem com aproximadamente uma semana de defasagem.

O PPI apresenta correlações superiores às do Brent no lag 1 para ambos os combustíveis, com destaque para o diesel. Esse comportamento é compatível com uma associação temporal entre alterações nos indicadores de preços ao produtor e alterações posteriores nos preços observados ao consumidor.

Esses resultados devem ser interpretados como associações estatísticas, e não como evidência de causalidade. Além disso, as análises envolvendo os preços da ANP estão restritas à janela temporal comum disponível entre as bases, do final de 2024 a agosto de 2026.